<a href="https://colab.research.google.com/github/Shravani2712/zomato-data-analysis-project/blob/main/Zomato%20Project%20Analysis%20in%20Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [3]:
file_path = "/content/Zomata - Copy.xlsx"
xls = pd.ExcelFile(file_path)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Zomata - Copy.xlsx'

In [ ]:
fact_main = pd.read_excel(xls, sheet_name="Main")
dim_country = pd.read_excel(xls, sheet_name="Country")
dim_calendar = pd.read_excel(xls, sheet_name="Calendar")
dim_currency = pd.read_excel(xls, sheet_name="Currency")

In [ ]:
fact_main = fact_main.merge(dim_country,how="left",left_on="CountryCode", right_on="CountryID")

In [ ]:
fact_main = fact_main.merge(dim_calendar,how="left",left_on="Datekey_Opening",right_on="Datekey_Opening")

In [ ]:
fact_main = fact_main.merge(dim_currency,how="left",on="Currency")

In [ ]:
fact_main.head()

In [ ]:
dim_currency.head()

In [ ]:
merged_df = fact_main.merge(dim_currency,how="left",on="Currency")

In [ ]:
merged_df.columns

In [ ]:
merged_df["Average_Cost_for_two_USD"] = merged_df["Average_Cost_for_two"] * merged_df["USD Rate_y"]
merged_df[["RestaurantName", "Currency", "Average_Cost_for_two", "USD Rate_y", "Average_Cost_for_two_USD"]].head()

In [ ]:
final_df = merged_df.copy()
final_df["Average_Cost_for_two_USD"] = final_df["Average_Cost_for_two_USD"].round(2)

In [ ]:
final_df = final_df.dropna(subset=["Average_Cost_for_two_USD"])

In [ ]:
restaurants_by_city_country = (final_df.groupby(["Countryname", "City"]).agg(Restaurant_Count=("RestaurantID", "count")).reset_index().sort_values("Restaurant_Count", ascending=False))

In [ ]:
restaurants_by_city_country.head(10)

In [ ]:
from matplotlib import pyplot as plt
_df_0['Restaurant_Count'].plot(kind='hist', bins=20, title='Restaurant_Count')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
top_cities = (
    final_df.groupby("City")
      .agg(Restaurant_Count=("RestaurantID", "count"))
      .reset_index()
      .sort_values("Restaurant_Count", ascending=False)
      .head(10)
)

In [ ]:
import matplotlib.pyplot as plt

top_cities.plot(
    x="City",
    y="Restaurant_Count",
    kind="bar",
    legend=False,
    title="Top 10 Cities by Number of Restaurants"
)
plt.ylabel("Number of Restaurants")
plt.show()


In [ ]:
restaurants_by_year = (
    final_df.groupby("Year")
      .agg(Restaurant_Count=("RestaurantID", "count"))
      .reset_index()
      .sort_values("Year")
)

In [ ]:
restaurants_by_quarter = (
    final_df.groupby(["Year", "Quarter"])
      .agg(Restaurant_Count=("RestaurantID", "count"))
      .reset_index()
      .sort_values(["Year", "Quarter"])
)

In [ ]:
restaurants_by_month = (
    final_df.groupby(["Year", "MonthName"])
      .agg(Restaurant_Count=("RestaurantID", "count"))
      .reset_index()
)

In [ ]:
month_order = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]

restaurants_by_month["MonthName"] = pd.Categorical(
    restaurants_by_month["MonthName"],
    categories=month_order,
    ordered=True
)

restaurants_by_month = restaurants_by_month.sort_values(["Year", "MonthName"]) # Sort by MonthName after converting to categorical

In [ ]:
import matplotlib.pyplot as plt

restaurants_by_year.plot(
    x="Year",
    y="Restaurant_Count",
    kind="bar",
    title="Restaurants Opening by Year",
    legend=False
)
plt.ylabel("Number of Restaurants")
plt.show()

In [ ]:
ratings_df = fact_main[
    (fact_main["Rating"].notna()) &
    (fact_main["Rating"] > 0)
]

In [ ]:
restaurants_by_rating = (
    ratings_df
    .groupby("Rating")
    .agg(Restaurant_Count=("RestaurantID", "count"))
    .reset_index()
    .sort_values("Rating")
)

In [ ]:
restaurants_by_rating.head(10)

In [ ]:
from matplotlib import pyplot as plt
_df_2['Rating'].plot(kind='hist', bins=20, title='Rating')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
bins = [0, 2, 3, 4, 5]
labels = ["Poor", "Average", "Good", "Excellent"]

ratings_df["Rating_Bucket"] = pd.cut(
    ratings_df["Rating"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

rating_bucket_count = (
    ratings_df
    .groupby("Rating_Bucket")
    .agg(Restaurant_Count=("RestaurantID", "count"))
    .reset_index()
)

In [ ]:
import matplotlib.pyplot as plt

rating_bucket_count.plot(
    x="Rating_Bucket",
    y="Restaurant_Count",
    kind="bar",
    legend=False,
    title="Restaurants by Rating Category"
)
plt.ylabel("Number of Restaurants")
plt.show()

In [ ]:
price_df = final_df[
    (final_df["Average_Cost_for_two_USD"].notna()) &
    (final_df["Average_Cost_for_two_USD"] > 0)
]

In [ ]:
bins = [0, 10, 30, 60, 100, price_df["Average_Cost_for_two_USD"].max()]
labels = ["Budget", "Affordable", "Mid-Range", "Premium", "Luxury"]

price_df["Price_Bucket"] = pd.cut(
    price_df["Average_Cost_for_two_USD"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
price_bucket_count = (
    price_df
    .groupby("Price_Bucket")
    .agg(Restaurant_Count=("RestaurantID", "count"))
    .reset_index()
)

In [ ]:
price_bucket_count

In [ ]:
from matplotlib import pyplot as plt
price_bucket_count['Restaurant_Count'].plot(kind='line', figsize=(8, 4), title='Restaurant_Count')
plt.gca().spines[['top', 'right']].set_visible(False)

In [ ]:
price_df["Dynamic_Price_Bucket"] = pd.qcut(
    price_df["Average_Cost_for_two_USD"],
    q=5,
    labels=["Very Cheap", "Cheap", "Moderate", "Expensive", "Very Expensive"]
)

In [ ]:
price_df

In [ ]:
booking_df = fact_main[fact_main["Has_Table_booking"].notna()]

In [ ]:
booking_counts = (
    booking_df
    .groupby("Has_Table_booking")
    .agg(Restaurant_Count=("RestaurantID", "count"))
    .reset_index()
)

In [ ]:
total_restaurants = booking_counts["Restaurant_Count"].sum()

booking_counts["Percentage"] = (
    booking_counts["Restaurant_Count"] / total_restaurants * 100
).round(2)

In [ ]:
booking_counts

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
booking_counts.groupby('Has_Table_booking').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
delivery_df = fact_main[fact_main["Has_Online_delivery"].notna()]

In [ ]:
delivery_counts = (
    delivery_df
    .groupby("Has_Online_delivery")
    .agg(Restaurant_Count=("RestaurantID", "count"))
    .reset_index()
)

In [ ]:
total_restaurants = delivery_counts["Restaurant_Count"].sum()

delivery_counts["Percentage"] = (
    delivery_counts["Restaurant_Count"] / total_restaurants * 100
).round(2)

In [ ]:
delivery_counts

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
delivery_counts.groupby('Has_Online_delivery').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
top_cuisines = (
    final_df["Cuisines"]
    .str.split(", ")
    .explode()
    .value_counts()
    .head(10)
)

top_cuisines.plot(kind="bar", title="Top 10 Cuisines by Restaurant Count")
plt.ylabel("Number of Restaurants")
plt.show()

In [ ]:
top_cities = (
    final_df.groupby("City")
      .agg(Restaurant_Count=("RestaurantID", "count"))
      .sort_values("Restaurant_Count", ascending=False)
      .head(10)
)

top_cities.plot(kind="bar", title="Top 10 Cities by Restaurants", legend=False)
plt.ylabel("Number of Restaurants")
plt.show()

In [ ]:
ratings_df = final_df[final_df["Rating"] > 0]

ratings_df["Rating"].plot(
    kind="hist",
    bins=10,
    title="Ratings Distribution"
)
plt.xlabel("Rating")
plt.show()

In [ ]:
avg_rating_city = (
    ratings_df.groupby("City")
    .agg(Avg_Rating=("Rating", "mean"))
    .sort_values("Avg_Rating", ascending=False)
    .head(10)
)

avg_rating_city.plot(kind="bar", title="Top Cities by Average Rating", legend=False)
plt.ylabel("Average Rating")
plt.show()

In [ ]:
delivery_country = (
    final_df.groupby(["Countryname", "Has_Online_delivery"])
    .size()
    .unstack()
)

delivery_country["Online_Delivery_%"] = (
    delivery_country["Yes"] /
    (delivery_country["Yes"] + delivery_country["No"]) * 100
)

(delivery_country["Online_Delivery_%"]
.sort_values(ascending=False)
.head(10)
.plot(
    kind="bar",
    title="Online Delivery Adoption by Country"
))
plt.ylabel("Percentage")
plt.show()

In [ ]:
price_rating = final_df[
    (final_df["Average_Cost_for_two_USD"] > 0) & (final_df["Rating"] > 0)
]

plt.scatter(
    price_rating["Average_Cost_for_two_USD"],
    price_rating["Rating"],
    alpha=0.4
)
plt.title("Price vs Rating Relationship")
plt.xlabel("Average Cost for Two (USD)")
plt.ylabel("Rating")
plt.show()

In [ ]:
total_restaurants = final_df["RestaurantID"].nunique()
avg_rating = ratings_df["Rating"].mean()
online_delivery_pct = (
    (final_df["Has_Online_delivery"] == "Yes").mean() * 100
)
table_booking_pct = (
    (final_df["Has_Table_booking"] == "Yes").mean() * 100
)

print("------ ZOMATO DASHBOARD KPIs ------")
print(f"Total Restaurants: {total_restaurants}")
print(f"Average Rating: {avg_rating:.2f}")
print(f"Online Delivery %: {online_delivery_pct:.2f}%")
print(f"Table Booking %: {table_booking_pct:.2f}%")